# Potato Harvest Detection — VDBorne Real Data
**BrabantHack 26 | Track Plant-Based 2 | Team 46**

## Pipeline
1. Load real images from `vdb/` (27k originals + 20k predictions)
2. Inspect existing classifier output (green = potato, blue = contaminant)
3. Build 3-class pseudo-label dataset: `potato`, `stone`, `stick`
4. Train YOLOv11s on real conveyor belt images
5. Per-frame counts → GPS contaminant zone map
6. Export GeoJSON + NDJSON


## 1. Install Dependencies

In [ ]:
!pip install ultralytics albumentations geopandas shapely scikit-learn \
             matplotlib seaborn torch torchvision opencv-python-headless -q


## 2. Paths & Config

In [ ]:
from pathlib import Path
import os, random, json, shutil
import numpy as np
import cv2
import matplotlib.pyplot as plt

ROOT        = Path('..')
ORIG_DIR    = ROOT / 'notebooks/data/originals'
PRED_DIR    = ROOT / 'notebooks/data/predictions'
DATA_DIR    = ROOT / 'notebooks/data/yolo'
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

# 3 classes
CLASSES = ['potato', 'stone', 'stick']
NC = len(CLASSES)

# Existing classifier bbox colors (BGR)
COLOR_POTATO      = (0, 255, 0)   # green → potato
COLOR_CONTAMINANT = (255, 0, 0)   # blue  → stone (manual labels will split stone/stick)

print(f'Originals   : {len(list(ORIG_DIR.glob("*.jpg")))}')
print(f'Predictions : {len(list(PRED_DIR.glob("*.jpg")))}')
print(f'Classes     : {CLASSES}')


## 3. Inspect Existing Classifier Output

Green boxes = potatoes.  Blue boxes = contaminants (stone or stick — to be split by manual labeling).


In [ ]:
def count_color(img, color_bgr, tol=40):
    lo = np.array([max(0, c-tol) for c in color_bgr], np.uint8)
    hi = np.array([min(255, c+tol) for c in color_bgr], np.uint8)
    return int(cv2.inRange(img, lo, hi).sum() / 255)

def show_pairs(n=6):
    preds = random.sample(list(PRED_DIR.glob('*.jpg')), n)
    fig, axes = plt.subplots(n, 2, figsize=(14, n*3))
    for i, pred_path in enumerate(preds):
        orig_name = pred_path.name.replace('-prediction-', '-picture-')
        orig_path = ORIG_DIR / orig_name
        if not orig_path.exists(): continue
        img_o = cv2.cvtColor(cv2.imread(str(orig_path)), cv2.COLOR_BGR2RGB)
        img_p = cv2.cvtColor(cv2.imread(str(pred_path)), cv2.COLOR_BGR2RGB)
        g = count_color(cv2.imread(str(pred_path)), COLOR_POTATO)
        b = count_color(cv2.imread(str(pred_path)), COLOR_CONTAMINANT)
        axes[i,0].imshow(img_o); axes[i,0].set_title('Original', fontsize=9)
        axes[i,1].imshow(img_p)
        axes[i,1].set_title(f'Prediction  potatoes={g}px  contaminants={b}px', fontsize=9)
        for ax in axes[i]: ax.axis('off')
    plt.suptitle('Original vs Existing Classifier', fontsize=13)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'pair_inspection.png', dpi=100)
    plt.show()

show_pairs(6)


## 4. Pseudo-Label Extraction

| Class | id | Source |
|---|---|---|
| `potato` | 0 | green bboxes from existing classifier |
| `stone`  | 1 | blue bboxes — **refine with manual labels** |
| `stick`  | 2 | blue bboxes — **refine with manual labels** |

Both `stone` and `stick` start as class `1` (stone) from the blue bbox.  
After manual labeling, re-run this cell with the corrected labels merged in.


In [ ]:
def extract_pseudo_labels(pred_img_bgr, img_w, img_h):
    labels = []
    for cls_id, color in [(0, COLOR_POTATO), (1, COLOR_CONTAMINANT)]:
        lo = np.array([max(0, c-40) for c in color], np.uint8)
        hi = np.array([min(255, c+40) for c in color], np.uint8)
        mask = cv2.inRange(pred_img_bgr, lo, hi)
        kernel = np.ones((5,5), np.uint8)
        mask = cv2.dilate(mask, kernel, iterations=2)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for c in contours:
            x, y, w, h = cv2.boundingRect(c)
            if w < 15 or h < 15: continue
            cx = (x + w/2) / img_w
            cy = (y + h/2) / img_h
            nw, nh = w / img_w, h / img_h
            labels.append(f'{cls_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
    return labels


SPLITS = {'train': 0.70, 'val': 0.20, 'test': 0.10}
MAX_IMAGES = 3000

for split in SPLITS:
    (DATA_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (DATA_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

all_preds = list(PRED_DIR.glob('*.jpg'))
random.shuffle(all_preds)
all_preds = all_preds[:MAX_IMAGES]

n = len(all_preds)
cut1 = int(n * SPLITS['train'])
cut2 = cut1 + int(n * SPLITS['val'])
split_map = ['train']*cut1 + ['val']*(cut2-cut1) + ['test']*(n-cut2)

written = {'train': 0, 'val': 0, 'test': 0}
for pred_path, split in zip(all_preds, split_map):
    orig_name = pred_path.name.replace('-prediction-', '-picture-')
    orig_path = ORIG_DIR / orig_name
    if not orig_path.exists(): continue
    pred_bgr = cv2.imread(str(pred_path))
    if pred_bgr is None: continue
    h, w = pred_bgr.shape[:2]
    labels = extract_pseudo_labels(pred_bgr, w, h)
    if not labels: continue
    stem = pred_path.stem.replace('-prediction-', '-')
    shutil.copy2(orig_path, DATA_DIR / split / 'images' / f'{stem}.jpg')
    (DATA_DIR / split / 'labels' / f'{stem}.txt').write_text('\n'.join(labels))
    written[split] += 1

print('Pseudo-label dataset:')
for s, c in written.items(): print(f'  {s}: {c} images')


## 5. data.yaml

In [ ]:
yaml_content = f"""path: {DATA_DIR.resolve()}
train: train/images
val:   val/images
test:  test/images
nc: {NC}
names: {CLASSES}
"""
DATASET_YAML = DATA_DIR / 'data.yaml'
DATASET_YAML.write_text(yaml_content)
print(yaml_content)


## 6. Augmentation Pipeline

In [ ]:
import albumentations as A

augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.MotionBlur(blur_limit=(3,9), p=0.4),
    A.RandomBrightnessContrast(0.35, 0.25, p=0.6),
    A.RandomGamma(gamma_limit=(75,125), p=0.3),
    A.CLAHE(clip_limit=3.0, p=0.2),
    A.GaussNoise(std_range=(0.02,0.1), p=0.35),
    A.HueSaturationValue(15, 25, 15, p=0.4),
    A.CoarseDropout(num_holes_range=(1,5),
                    hole_height_range=(10,35),
                    hole_width_range=(10,35), p=0.25),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.3))

print('Augmentation pipeline ready')


## 7. YOLOv11s Training

Classes: `potato` · `stone` · `stick`  
Set `EPOCHS=5` for quick demo, `EPOCHS=50` for real training.


In [ ]:
from ultralytics import YOLO
import torch

print(f'PyTorch: {torch.__version__}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model = YOLO('yolo11s.pt')

EPOCHS = 5  # ← set to 50 for real training

results = model.train(
    data=str(DATASET_YAML),
    epochs=EPOCHS,
    imgsz=640,
    batch=8,
    device=device,
    mosaic=0.8,
    degrees=5,
    flipud=0.3,
    fliplr=0.5,
    lr0=0.01,
    lrf=0.005,
    project=str(RESULTS_DIR),
    name='yolo11s_3class',
    exist_ok=True,
    verbose=True,
)

BEST_MODEL = Path(results.save_dir) / 'weights/best.pt'
print(f'Best model: {BEST_MODEL}')


## 8. Inference — Per-Frame Counts

Output: potato count, stone count, stick count per frame.


In [ ]:
from ultralytics import YOLO
import pandas as pd

infer_model = YOLO(str(BEST_MODEL))
test_imgs = list((DATA_DIR / 'test/images').glob('*.jpg'))
print(f'Inference on {len(test_imgs)} test images...')

records = []
for img_path in test_imgs:
    result = infer_model(str(img_path), verbose=False)[0]
    counts = {'potato': 0, 'stone': 0, 'stick': 0}
    for box in result.boxes:
        cls_name = CLASSES[int(box.cls)]
        counts[cls_name] += 1
    parts = img_path.stem.split('__')
    cam = parts[0] if len(parts) > 1 else 'unknown'
    ts  = parts[1][:19] if len(parts) > 1 else img_path.stem[:19]
    records.append({'cam': cam, 'ts': ts, **counts})

det_df = pd.DataFrame(records)
det_df['contaminants'] = det_df['stone'] + det_df['stick']
det_df['contaminant_rate'] = (det_df['contaminants'] /
                               det_df[['potato','contaminants']].sum(axis=1).replace(0,1) * 100).round(1)

print(det_df[['potato','stone','stick','contaminant_rate']].describe().round(2))
print(f'\nFrames with contaminants : {(det_df.contaminants > 0).sum()}/{len(det_df)}')
print(f'Avg contaminant rate     : {det_df.contaminant_rate.mean():.1f}%')


## 9. GPS Contaminant Zone Map

Clusters frames with high contaminant rates into field risk zones.


In [ ]:
from shapely.geometry import Point, MultiPoint
from sklearn.cluster import DBSCAN

# Simulate GPS track — replace with real GPS CSV if available
# gps_df = pd.read_csv('../data/gps_track.csv')  # cols: ts, lat, lon
np.random.seed(42)
n = len(det_df)
base_lat, base_lon = 51.350, 5.150
lats, lons = [], []
for row in range(10):
    row_lons = np.linspace(base_lon, base_lon + 0.003, n // 10 + 1)
    if row % 2: row_lons = row_lons[::-1]
    row_lat = base_lat + row * 0.0002
    for lon in row_lons[:n // 10]:
        lats.append(row_lat + np.random.normal(0, 0.00001))
        lons.append(lon  + np.random.normal(0, 0.00001))
det_df = det_df.iloc[:len(lats)].copy()
det_df['lat'] = lats[:len(det_df)]
det_df['lon'] = lons[:len(det_df)]

# DBSCAN on high-contaminant frames
high = det_df[det_df.contaminant_rate > 15].copy()
zones = []
if len(high) > 3:
    lbls = DBSCAN(eps=0.0001, min_samples=3).fit_predict(high[['lat','lon']].values)
    high['zone_id'] = lbls
    for lbl in set(lbls):
        if lbl == -1: continue
        cluster = high[high.zone_id == lbl]
        if len(cluster) < 3: continue
        pts = MultiPoint([(lo, la) for la, lo in cluster[['lat','lon']].values])
        poly = pts.convex_hull.buffer(0.00015)
        avg_rate  = cluster.contaminant_rate.mean()
        avg_stone = cluster.stone.mean()
        avg_stick = cluster.stick.mean()
        zones.append({
            'zone_id': f'zone_{lbl+1}',
            'risk': 'high' if avg_rate > 30 else 'medium',
            'avg_contaminant_rate': round(avg_rate, 1),
            'avg_stones': round(avg_stone, 1),
            'avg_sticks': round(avg_stick, 1),
            'n_frames': len(cluster),
            'geometry': poly,
        })
    print(f'Detected {len(zones)} contaminant zone(s)')
    for z in zones:
        print(f"  {z['zone_id']}: {z['risk']}, {z['avg_contaminant_rate']}% contaminant, {z['n_frames']} frames")
else:
    print('Not enough high-contaminant frames for clustering')


## 10. Visualise

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Contaminant rate map
sc = axes[0].scatter(det_df.lon, det_df.lat, c=det_df.contaminant_rate,
                     cmap='OrRd', s=20, alpha=0.8, vmin=0, vmax=50)
plt.colorbar(sc, ax=axes[0], label='Contaminant rate %')
for z in zones:
    x, y = z['geometry'].exterior.xy
    col = '#cc0000' if z['risk'] == 'high' else '#ff8800'
    axes[0].fill(x, y, alpha=0.25, color=col)
    axes[0].plot(x, y, color=col, lw=2)
    axes[0].annotate(z['zone_id'], (float(np.mean(x)), float(np.mean(y))),
                     fontsize=8, ha='center', color=col, fontweight='bold')
axes[0].set_title('Contaminant Rate per Zone', fontsize=12)
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

# Stone vs stick breakdown
sc2 = axes[1].scatter(det_df.lon, det_df.lat, c=det_df.stone,
                      cmap='Blues', s=20, alpha=0.8)
plt.colorbar(sc2, ax=axes[1], label='Stones / frame')
axes[1].set_title('Stone Density', fontsize=12)
axes[1].set_xlabel('Longitude')

# Contaminant count over track
axes[2].bar(range(len(det_df)), det_df.stone, label='Stone', color='#5b87c5', alpha=0.8)
axes[2].bar(range(len(det_df)), det_df.stick, bottom=det_df.stone,
            label='Stick', color='#8B5E3C', alpha=0.8)
axes[2].axhline(3, color='orange', ls='--', lw=1.5, label='Warning (3/frame)')
axes[2].set_xlabel('Frame'); axes[2].set_ylabel('Contaminants')
axes[2].set_title('Stone + Stick Count Along Track', fontsize=12)
axes[2].legend(fontsize=9)

plt.suptitle('VDBorne — Harvest Contaminant Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'contaminant_map.png', dpi=120)
plt.show()
print('Saved: results/contaminant_map.png')


## 11. Export — GeoJSON + NDJSON

In [ ]:
from shapely.geometry import mapping

if zones:
    geojson = {'type': 'FeatureCollection', 'features': [
        {'type': 'Feature',
         'geometry': mapping(z['geometry']),
         'properties': {k: v for k, v in z.items() if k != 'geometry'}}
        for z in zones
    ]}
    (RESULTS_DIR / 'contaminant_zones.geojson').write_text(json.dumps(geojson, indent=2))
    print(f'GeoJSON: {len(zones)} zones')

ndjson_path = RESULTS_DIR / 'detections_batch.ndjson'
with open(ndjson_path, 'w') as f:
    for r in det_df.to_dict('records'):
        f.write(json.dumps(r) + '\n')
print(f'NDJSON: {len(det_df)} records → {ndjson_path}')

print('\n=== KPI Summary ===')
print(f'Frames analysed          : {len(det_df)}')
print(f'Avg potatoes / frame     : {det_df.potato.mean():.1f}')
print(f'Avg stones / frame       : {det_df.stone.mean():.1f}')
print(f'Avg sticks / frame       : {det_df.stick.mean():.1f}')
print(f'Avg contaminant rate     : {det_df.contaminant_rate.mean():.1f}%')
print(f'High-risk zones          : {sum(z["risk"]=="high" for z in zones)}')
